In [3]:
import pandas as pd

# 读取信用审批数据
file_path = "信用审批Case筛选及审批记录导出需求_202606.xlsx"
df = pd.read_excel(file_path)

# 获取所有唯一Case编号
all_ids = df['参与方验证编码'].unique().tolist()
total_cases = len(all_ids)

# 筛选经过"财务二级审核"且审批通过的Case（财务二级审核同意 = 信用审核通过）
fin2 = df[df['信审审批节点名称'] == '04 财务二级审核']

approved_cases = []
for app_id in all_ids:
    case_rows = fin2[fin2['参与方验证编码'] == app_id]
    if len(case_rows) == 0:
        continue
    
    row = case_rows.iloc[0]
    create_time = pd.to_datetime(df[df['参与方验证编码'] == app_id]['创建时间'].iloc[0])
    complete_time = pd.to_datetime(row['审批完成时间'])
    duration = (complete_time - create_time).days
    
    approved_cases.append({
        '编号': app_id,
        '申请人': df[df['参与方验证编码'] == app_id]['信审申请人'].iloc[0],
        '客户': df[df['参与方验证编码'] == app_id]['信审客户-iBOSS名称'].iloc[0],
        '用时(天)': duration
    })

approved_df = pd.DataFrame(approved_cases)

# 统计
min_case = approved_df.loc[approved_df['用时(天)'].idxmin()]
max_case = approved_df.loc[approved_df['用时(天)'].idxmax()]
avg_days = approved_df['用时(天)'].mean()

print("=" * 70)
print("📊 信用审核Case统计概览")
print("=" * 70)
print(f"  申请总数:          {total_cases} 个Case")
print(f"  通过审核(财务二级): {len(approved_cases)} 个Case")
print(f"  未通过/进行中:     {total_cases - len(approved_cases)} 个Case")
print("-" * 70)
print(f"  ⚡ 最快通过: {min_case['编号']}  用时 {min_case['用时(天)']} 天  ({min_case['客户']})")
print(f"  🐢 最慢通过: {max_case['编号']}  用时 {max_case['用时(天)']} 天  ({max_case['客户']})")
print(f"  📈 平均用时: {avg_days:.1f} 天")
print("=" * 70)

📊 信用审核Case统计概览
  申请总数:          41 个Case
  通过审核(财务二级): 24 个Case
  未通过/进行中:     17 个Case
----------------------------------------------------------------------
  ⚡ 最快通过: FV-20260409-51542  用时 1 天  (SANHUA AUTOMOTIVE USA，LLC)
  🐢 最慢通过: FV-20260319-51126  用时 20 天  (SECURITY NETWORK INTELLIGENCE CLOUD LIMITED)
  📈 平均用时: 7.6 天


In [79]:
# ============================================
# 功能说明：押金相关Case统计与跟踪
# ============================================
# 1. 识别明确要求缴纳押金的case（排除豁免和建议性语言）
# 2. 检查是否已上传押金凭证（财务部确认到款）
# 3. 提取销售信息（销售姓名、销售部门、客户名称）
# 4. 显示财务要求缴纳押金的日期
# 5. 显示所有押金case的详细信息
# 6. 标注需要跟进的case（未上传押金凭证）
# ============================================

# 押金相关统计
deposit_cases = []
deposit_with_receipt = []
deposit_without_receipt = []

for app_id in all_ids:
    app_df = df[df['参与方验证编码'] == app_id]
    finance_1st_all = app_df[app_df['信审审批节点名称'] == '03 财务一级审核']
    
    is_require_deposit = False
    head_opinion = '无'
    deposit_date = None
    
    if len(finance_1st_all) > 0:
        for idx, row in finance_1st_all.iterrows():
            opinion_1st = str(row['审批意见']).lower() if pd.notna(row['审批意见']) else ''
            is_exempt = '豁免押金' in opinion_1st or '免押金' in opinion_1st
            
            if (
                ('客户需缴纳押金' in opinion_1st or 
                 '需缴纳押金' in opinion_1st or 
                 '要求缴纳押金' in opinion_1st or
                 '在收到' in opinion_1st and '押金' in opinion_1st) and not is_exempt
            ):
                is_require_deposit = True
                head_opinion = row['审批意见']
                # 获取财务要求缴纳押金的日期
                deposit_date = row['审批开始时间'] if pd.notna(row['审批开始时间']) else None
                break
        
        if not is_require_deposit:
            head_opinion = finance_1st_all.iloc[0]['审批意见']
        
        if is_require_deposit:
            has_receipt = len(app_df[app_df['信审审批节点名称'] == '18 财务部（合同组）确认到款']) > 0
            sales_name = app_df['信审申请人'].iloc[0] if '信审申请人' in app_df.columns else '未知'
            sales_dept = app_df['信审申请部门'].iloc[0] if '信审申请部门' in app_df.columns else '未知'
            customer = app_df['信审客户-iBOSS名称'].iloc[0] if '信审客户-iBOSS名称' in app_df.columns else '未知'
            
            # 格式化日期
            deposit_date_str = deposit_date.strftime('%Y-%m-%d') if deposit_date is not None else '未知'
            
            deposit_info = {
                '编号': app_id,
                '销售': sales_name,
                '销售部门': sales_dept,
                '客户': customer,
                '押金要求日期': deposit_date_str,
                '总部意见': str(head_opinion)[:100] + '...' if len(str(head_opinion)) > 100 else str(head_opinion),
                '已上传凭证': '是' if has_receipt else '否'
            }
            deposit_cases.append(deposit_info)
            
            if has_receipt:
                deposit_with_receipt.append(deposit_info)
            else:
                deposit_without_receipt.append(deposit_info)

# 显示押金统计
print("=" * 80)
print("押金相关Case统计")
print("=" * 80)
print(f"明确要求缴纳押金的Case总数: {len(deposit_cases)}")
print(f"已上传押金凭证: {len(deposit_with_receipt)}")
print(f"未上传押金凭证: {len(deposit_without_receipt)}")

if deposit_cases:
    print(f"\n📋 所有押金Case详细信息:")
    print("-" * 80)
    for i, case in enumerate(deposit_cases, 1):
        print(f"\n【Case {i}】")
        print(f"  编号: {case['编号']}")
        print(f"  销售: {case['销售']} ({case['销售部门']})")
        print(f"  客户: {case['客户']}")
        print(f"  押金要求日期: {case['押金要求日期']}")
        print(f"  上传状态: {case['已上传凭证']}")
        if case['已上传凭证'] == '否':
            print(f"  ⚠️  需要跟进！")
        print(f"  总部意见: {case['总部意见']}")
else:
    print("\n没有需要缴纳押金的Case")

print("=" * 80)

押金相关Case统计
明确要求缴纳押金的Case总数: 2
已上传押金凭证: 1
未上传押金凭证: 1

📋 所有押金Case详细信息:
--------------------------------------------------------------------------------

【Case 1】
  编号: FV-20260423-51823
  销售: robertsong (美国公司-企业业务)
  客户: ATTO DIGITAL INFORMATION TECHNOLOGY (HONG KONG) LIMITED
  押金要求日期: 2026-04-27
  上传状态: 否
  ⚠️  需要跟进！
  总部意见: 根据第三方信用报告评估结果，建议客户为中风险，在收到不少于50%的一次性费用及相等于2个月月费的押金(可接受形式有：现金、Traffic Prepayment)之后，授予客户相等于2个月月费的信用额开...

【Case 2】
  编号: FV-20260606-52662
  销售: christineluo (美国公司-企业业务)
  客户: 北京数信网联信息科技有限公司
  押金要求日期: 2026-06-11
  上传状态: 是
  总部意见: 客户需缴纳押金才能开展业务，外汇管制风险可控，经核对，拟同意
